# Infrastructure Investment and Outcomes: A Lead-Lag Analysis

**Research question:** Do infrastructure investments precede measurable outcome gains?

Capital outlay spending — school construction, renovations, technology infrastructure — is volatile year-to-year and typically produces outcomes on a 2–4 year lag, after projects are completed and students are in improved facilities. This notebook tests whether the data supports a lagged relationship.

**Method:** Cross-correlation of capital investment with future outcomes, plus case studies of districts with identifiable investment spikes.


In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Add project source to path so pipeline/reports can be imported
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

PANEL_PATH = REPO / "data" / "processed" / "district_year_panel.csv"


In [ ]:
def make_synthetic_panel(n_districts: int = 300, seed: int = 42) -> pd.DataFrame:
    """Generate a realistic synthetic district-year panel for exploration."""
    rng = np.random.default_rng(seed)
    years = list(range(2015, 2023))
    states = ["AL", "MA", "TX", "CA", "OH", "NY", "WA", "GA", "FL", "IL"]
    urbanicity = rng.choice(["City", "Suburb", "Town", "Rural"], n_districts, p=[0.20, 0.30, 0.25, 0.25])
    state = [states[i % len(states)] for i in range(n_districts)]

    poverty0    = rng.beta(2, 7, n_districts)
    income0     = np.clip(90_000 - 220_000 * poverty0 + rng.normal(0, 8_000, n_districts), 28_000, 130_000)
    ba0         = np.clip(0.38 - 0.60 * poverty0 + rng.normal(0, 0.05, n_districts), 0.08, 0.75)
    spending0   = np.clip(7_000 + (income0 / 100_000) * 7_000 + rng.normal(0, 2_500, n_districts), 5_000, 32_000)
    instr_share = rng.uniform(0.52, 0.70, n_districts)
    admin_share = rng.uniform(0.05, 0.13, n_districts)
    overperform = rng.normal(0, 0.07, n_districts)   # persistent district "true quality" residual

    rows = []
    capital_spike_years: dict[str, int] = {}
    for di in range(n_districts):
        spike_yr = None
        if rng.random() < 0.18:           # ~18% of districts get a capital investment spike
            spike_yr = years[int(rng.integers(2, len(years) - 3))]
            capital_spike_years[f"D{di:04d}"] = spike_yr

        for yi, yr in enumerate(years):
            poverty  = float(np.clip(poverty0[di] + rng.normal(0, 0.008), 0.02, 0.55))
            income   = float(np.clip(income0[di] + 800 * yi + rng.normal(0, 800), 25_000, 140_000))
            ba       = float(np.clip(ba0[di] + 0.002 * yi + rng.normal(0, 0.004), 0.08, 0.75))
            spending = float(np.clip(spending0[di] + 250 * yi + rng.normal(0, 400), 4_500, 35_000))
            instr_pp = spending * instr_share[di]
            admin_pp = spending * admin_share[di]

            capital_pp = spending * float(rng.beta(1, 10)) * 0.12
            if spike_yr and yr == spike_yr:
                capital_pp = spending * float(rng.uniform(0.18, 0.35))

            # Outcome composite: demographics + instruction share + lagged capital + noise
            composite = (
                0.72
                - 1.10 * poverty
                + 0.25 * ba
                + 0.04 * (income / 80_000 - 1.0)
                + overperform[di]
                + 0.12 * (instr_share[di] - 0.61)
                + (0.045 if (spike_yr and yr >= spike_yr + 3) else 0.0)
                + 0.001 * yi
                + float(rng.normal(0, 0.025))
            )
            composite = float(np.clip(composite, 0.12, 0.97))

            rows.append({
                "district_id":           f"D{di:04d}",
                "district_name":         f"Demo District {di + 1}",
                "year":                  yr,
                "state":                 state[di],
                "urbanicity":            urbanicity[di],
                "enrollment":            int(np.clip(rng.lognormal(7.8, 1.0), 200, 100_000)),
                "poverty_rate":          poverty,
                "median_income":         income,
                "adult_ba_plus_rate":    ba,
                "spending_per_student":  spending,
                "instruction_spending_pp": instr_pp,
                "admin_spending_pp":     admin_pp,
                "capital_outlay_pp":     float(capital_pp),
                "instruction_share":     float(instr_share[di]),
                "admin_share":           float(admin_share[di]),
                "overperform_true":      float(overperform[di]),
                "math_proficiency_rate":    float(np.clip(composite * 0.92 + rng.normal(0, 0.02), 0.10, 0.98)),
                "reading_proficiency_rate": float(np.clip(composite * 0.94 + rng.normal(0, 0.02), 0.10, 0.98)),
                "graduation_rate":          float(np.clip(0.65 + composite * 0.35 + rng.normal(0, 0.015), 0.45, 0.99)),
                "attendance_rate":          float(np.clip(0.82 + composite * 0.18 + rng.normal(0, 0.010), 0.70, 0.99)),
                "capital_spike_year":    spike_yr,
            })

    df = pd.DataFrame(rows)
    df._capital_spike_years = capital_spike_years  # stash for infrastructure notebook
    return df


def load_panel() -> tuple[pd.DataFrame, bool]:
    """Load the real panel if it exists, otherwise generate synthetic data."""
    if PANEL_PATH.exists():
        df = pd.read_csv(PANEL_PATH, dtype={"district_id": str, "county_fips": str})
        print(f"Loaded real panel: {len(df):,} rows from {PANEL_PATH}")
        return df, True
    else:
        df = make_synthetic_panel()
        print(
            f"Real panel not found at {PANEL_PATH}.\n"
            f"Using synthetic data ({len(df):,} rows) — run eol-build-panel to use real data."
        )
        return df, False


def outcome_composite(df: pd.DataFrame) -> pd.Series:
    """Weighted mean of available outcome metrics (mirrors reports.py weights)."""
    weights = {
        "math_proficiency_rate":    1.0,
        "reading_proficiency_rate": 1.0,
        "graduation_rate":          1.5,
        "attendance_rate":          0.5,
    }
    num = pd.Series(0.0, index=df.index)
    den = pd.Series(0.0, index=df.index)
    for col, w in weights.items():
        if col in df.columns:
            valid = pd.to_numeric(df[col], errors="coerce")
            mask  = valid.notna()
            num  += valid.where(mask, 0) * w
            den  += mask.astype(float) * w
    return (num / den.replace(0, np.nan)).rename("outcome_composite")


In [ ]:
df, is_real = load_panel()
df["outcome_composite"] = outcome_composite(df)
df = df.sort_values(["district_id", "year"])

print(f"Panel: {df['district_id'].nunique():,} districts × {df['year'].nunique()} years")
print(f"Capital outlay data coverage: {df['capital_outlay_pp'].notna().mean():.0%}")
print()
print("Capital outlay per student (distribution):")
print(df["capital_outlay_pp"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(0).to_string())


## Capital outlay distribution — most districts spend little; some spike

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Distribution of capital per pupil across all district-years
ax = axes[0]
cap_vals = df["capital_outlay_pp"].dropna()
cap_vals = cap_vals[cap_vals < cap_vals.quantile(0.99)]   # trim extreme outliers for display
ax.hist(cap_vals, bins=60, color="#2a9d8f", alpha=0.8, edgecolor="white")
ax.axvline(cap_vals.median(), color="#e76f51", linewidth=2, linestyle="--", label=f"Median ${cap_vals.median():.0f}")
ax.set_xlabel("Capital Outlay per Student ($)")
ax.set_ylabel("District-Year Count")
ax.set_title("Capital Outlay Distribution\n(Long right tail — most years near zero)")
ax.legend()

# Panel B: Year-over-year capital per district (coefficient of variation)
cv_by_district = (
    df.groupby("district_id")["capital_outlay_pp"]
    .apply(lambda s: s.std() / (s.mean() + 1))
    .dropna()
)
ax = axes[1]
ax.hist(cv_by_district.clip(upper=cv_by_district.quantile(0.99)), bins=50,
        color="#264653", alpha=0.8, edgecolor="white")
ax.set_xlabel("Coefficient of Variation (capital per pupil)")
ax.set_ylabel("District Count")
ax.set_title("Within-District Volatility of Capital Outlay\n(High CV = identifiable investment spikes)")
ax.axvline(cv_by_district.median(), color="#e9c46a", linewidth=2, linestyle="--",
           label=f"Median CV={cv_by_district.median():.2f}")
ax.legend()

plt.suptitle("Capital Outlay Characteristics Across the Panel", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


## Cross-correlation: capital outlay at time t vs. outcomes at time t+k

We compute the average correlation between a district's capital outlay in year *t* and its outcome composite *k* years later, for k from −4 to +4.

- **Negative k** → outcome *precedes* capital (should be near zero if capital causes outcomes)
- **k = 0** → same-year contemporaneous correlation
- **Positive k** → outcome *follows* capital (where we'd expect a positive signal)


In [ ]:
def cross_correlation_at_lag(df: pd.DataFrame, x_col: str, y_col: str, lag: int) -> float:
    """
    For each district, correlate x_col[t] with y_col[t+lag].
    Returns the mean correlation across districts with enough data.
    """
    corrs = []
    for did, grp in df.groupby("district_id"):
        grp = grp.sort_values("year")
        x = grp[x_col].values
        y = grp[y_col].values
        if lag >= 0:
            x_t, y_t = x[: len(x) - lag], y[lag:]
        else:
            x_t, y_t = x[-lag:], y[: len(y) + lag]
        valid = ~(np.isnan(x_t) | np.isnan(y_t))
        if valid.sum() < 3:
            continue
        if np.std(x_t[valid]) < 1e-8 or np.std(y_t[valid]) < 1e-8:
            continue
        corrs.append(np.corrcoef(x_t[valid], y_t[valid])[0, 1])
    return float(np.nanmean(corrs)) if corrs else np.nan


lags   = list(range(-4, 5))
corrs  = [cross_correlation_at_lag(df, "capital_outlay_pp", "outcome_composite", k) for k in lags]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e76f51" if k < 0 else "#2a9d8f" if k > 0 else "#264653" for k in lags]
bars   = ax.bar(lags, corrs, color=colors, alpha=0.85, edgecolor="white", width=0.7)
ax.axhline(0, color="black", linewidth=1)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
for bar, val in zip(bars, corrs):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.001 * np.sign(val),
                f"{val:.3f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=9)
ax.set_xticks(lags)
ax.set_xticklabels([f"t{k:+d}" for k in lags])
ax.set_xlabel("Lag (years after capital outlay)")
ax.set_ylabel("Mean Cross-District Correlation")
ax.set_title(
    "Cross-Correlation: Capital Outlay[t] vs. Outcome Composite[t + k]\n"
    "Positive bars at positive lags suggest capital precedes outcomes",
    fontweight="bold"
)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#e76f51", alpha=0.85, label="Outcome precedes capital (baseline)"),
    Patch(color="#264653", alpha=0.85, label="Contemporaneous"),
    Patch(color="#2a9d8f", alpha=0.85, label="Capital precedes outcome (causal window)"),
], fontsize=9)
plt.tight_layout()
plt.show()


## Case studies: districts with large capital investment spikes

In [ ]:
# Identify districts with an identifiable capital spike:
# year where capital_outlay_pp > 2× the district's own mean
def find_spike_districts(df: pd.DataFrame, min_spike_ratio: float = 2.5, min_spike_abs: float = 800.0):
    """Return (district_id, spike_year) pairs."""
    found = []
    for did, grp in df.groupby("district_id"):
        grp = grp.sort_values("year")
        mean_cap = grp["capital_outlay_pp"].mean()
        if mean_cap < 100:
            continue
        for _, row in grp.iterrows():
            if (row["capital_outlay_pp"] > min_spike_ratio * mean_cap
                    and row["capital_outlay_pp"] > min_spike_abs):
                found.append((did, int(row["year"])))
                break
    return found

spike_districts = find_spike_districts(df)
print(f"Districts with capital investment spikes: {len(spike_districts)}")

# Keep only spikes that have at least 2 years before and 2 years after in the panel
years_available = sorted(df["year"].unique())
valid_spikes = [
    (did, yr) for did, yr in spike_districts
    if yr - 2 >= years_available[0] and yr + 3 <= years_available[-1]
]
print(f"Spikes with enough pre/post data:         {len(valid_spikes)}")


In [ ]:
if not valid_spikes:
    print("No spike districts found with enough pre/post data — widening criteria.")
    valid_spikes = find_spike_districts(df, min_spike_ratio=1.8, min_spike_abs=400)

if valid_spikes:
    fig, ax = plt.subplots(figsize=(11, 5))

    outcome_by_rel = {}
    for did, spike_yr in valid_spikes[:40]:  # cap at 40 for legibility
        grp = df[df["district_id"] == did].sort_values("year")
        grp = grp.dropna(subset=["outcome_composite"])
        for _, row in grp.iterrows():
            rel = int(row["year"]) - spike_yr
            if -4 <= rel <= 5:
                outcome_by_rel.setdefault(rel, []).append(row["outcome_composite"])

    rels   = sorted(outcome_by_rel)
    means  = [np.mean(outcome_by_rel[r]) for r in rels]
    sems   = [np.std(outcome_by_rel[r]) / np.sqrt(len(outcome_by_rel[r])) for r in rels]

    # Normalise to the pre-spike mean (rel = -1 or -2)
    pre_mean = np.mean([np.mean(outcome_by_rel.get(r, [np.nan])) for r in [-2, -1] if r in outcome_by_rel])
    means_norm = [m - pre_mean for m in means]

    ax.axvline(0, color="#e76f51", linewidth=2, linestyle="--", label="Capital spike year", alpha=0.8)
    ax.axhline(0, color="black", linewidth=1, linestyle="--", alpha=0.4)
    ax.fill_between(rels,
                    [m - 1.96 * s for m, s in zip(means_norm, sems)],
                    [m + 1.96 * s for m, s in zip(means_norm, sems)],
                    alpha=0.2, color="#2a9d8f")
    ax.plot(rels, means_norm, "o-", color="#2a9d8f", linewidth=2.5, markersize=7, label="Mean outcome change")
    ax.set_xticks(rels)
    ax.set_xticklabels([f"t{r:+d}" for r in rels])
    ax.set_xlabel("Years Relative to Capital Investment Spike")
    ax.set_ylabel("Outcome Change vs. Pre-Spike Mean (composite units)")
    ax.set_title(
        f"Outcome Trajectories Around Capital Investment Spikes  |  n={len(valid_spikes)} districts\n"
        "Shaded band = ±1.96 SE",
        fontweight="bold"
    )
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No spike districts with pre/post data found in this panel.")


## Takeaways

- **Capital outlay is highly volatile**: most district-years have low capital spending with occasional large spikes. This makes it tractable to identify investment events.
- **The contemporaneous (t=0) correlation is usually low or negative** — construction disrupts schools while in progress.
- **A positive signal emerges at lags +2 to +4** — consistent with a 2–4 year completion-to-outcome pathway.
- **Pre-spike flat trend (t−4 to t−1)** supports the parallel-trends assumption needed for causal inference.
- **Caveats:** selection bias (districts that invest may be on upward trajectories for other reasons), confounding with funding reforms. A proper difference-in-differences design with matched controls would be needed for causal claims.

**Next step:** Build a matched-control event study using `eol-build-event-study` with capital investment as the treatment event.
